# SteelFlow: ML experiment suite

This notebook compares multiple algorithms for demand forecasting, late-delivery classification, and freight-cost regression. It uses the supplied time-based splits and records metrics in one results table.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
con = duckdb.connect()
sns.set_theme(style="whitegrid")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.base import clone


## Late-delivery classification

In [ ]:
late = con.execute(f"SELECT * FROM read_parquet('{(DATA/'dispatch_delay_model_dataset.parquet').as_posix()}')").fetchdf()
target = 'target_is_late'
drop = [target, 'shipment_id', 'dispatch_timestamp', 'dispatch_date', 'prediction_moment', 'synthetic_record']
X = late.drop(columns=[c for c in drop if c in late]); y = late[target].astype(int)
cat = X.select_dtypes(include=['object','bool']).columns.tolist(); num = [c for c in X.columns if c not in cat]
pre = ColumnTransformer([('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num), ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
train, val, test = late['split'].eq('TRAIN'), late['split'].eq('VALIDATION'), late['split'].eq('TEST')
classifiers = {'logistic': LogisticRegression(max_iter=1000, class_weight='balanced'), 'random_forest': RandomForestClassifier(n_estimators=250, min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=42)}
classification_results = []
for name, estimator in classifiers.items():
    pipe = Pipeline([('pre', clone(pre)), ('model', estimator)])
    pipe.fit(X[train], y[train]); prob = pipe.predict_proba(X[val])[:, 1]
    classification_results.append({'task':'late_delivery','model':name,'pr_auc':average_precision_score(y[val],prob),'roc_auc':roc_auc_score(y[val],prob),'f1':f1_score(y[val],prob >= .5)})
pd.DataFrame(classification_results).sort_values('pr_auc', ascending=False)

## Freight-cost regression

In [ ]:
freight = con.execute(f"SELECT * FROM read_parquet('{(DATA/'freight_cost_model_dataset.parquet').as_posix()}')").fetchdf()
target = 'target_actual_billed_freight_cost_inr_ex_gst'
drop = [target, 'freight_invoice_id', 'shipment_id', 'quote_timestamp', 'prediction_moment', 'synthetic_record']
X = freight.drop(columns=[c for c in drop if c in freight]); y = freight[target]
cat = X.select_dtypes(include=['object','bool']).columns.tolist(); num = [c for c in X.columns if c not in cat]
pre_reg = ColumnTransformer([('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num), ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
train, val = freight['split'].eq('TRAIN'), freight['split'].eq('VALIDATION')
regressors = {'ridge': Ridge(alpha=10), 'random_forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42)}
regression_results = []
for name, estimator in regressors.items():
    pipe = Pipeline([('pre', clone(pre_reg)), ('model', estimator)])
    pipe.fit(X[train], y[train]); pred = pipe.predict(X[val])
    regression_results.append({'task':'freight_cost','model':name,'mae':mean_absolute_error(y[val],pred),'rmse':mean_squared_error(y[val],pred)**0.5})
pd.DataFrame(regression_results).sort_values('mae')

## Demand forecasting benchmarks

In [ ]:
h07 = con.execute(f"SELECT * FROM read_parquet('{(DATA/'forecast_features_horizon_07.parquet').as_posix()}')").fetchdf()
exclude = ['forecast_row_id','origin_date','prediction_moment','feature_cutoff_date','target_date','target_demand_kg','split','synthetic_record']
X = h07.drop(columns=[c for c in exclude if c in h07]); y = h07['target_demand_kg']
cat = X.select_dtypes(include=['object','bool']).columns.tolist(); num = [c for c in X.columns if c not in cat]
pre_dem = ColumnTransformer([('num', Pipeline([('impute', SimpleImputer(strategy='median'))]), num), ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
train, val = h07['split'].eq('TRAIN'), h07['split'].eq('VALIDATION')
regressors = {'ridge': Ridge(alpha=10), 'random_forest': RandomForestRegressor(n_estimators=150, min_samples_leaf=5, n_jobs=-1, random_state=42), 'hist_gradient_boosting': HistGradientBoostingRegressor(max_iter=150, learning_rate=.08, l2_regularization=1.0, random_state=42)}
demand_results = []
for name, estimator in regressors.items():
    pipe = Pipeline([('pre', clone(pre_dem)), ('model', estimator)])
    pipe.fit(X[train], y[train]); pred = np.maximum(pipe.predict(X[val]), 0)
    mae = mean_absolute_error(y[val], pred); wape = np.abs(y[val]-pred).sum()/np.abs(y[val]).sum()
    demand_results.append({'task':'demand_horizon_07','model':name,'mae_kg':mae,'rmse_kg':mean_squared_error(y[val],pred)**0.5,'wape':wape})
pd.DataFrame(demand_results).sort_values('wape')

In [ ]:
all_results = pd.concat([pd.DataFrame(classification_results), pd.DataFrame(regression_results), pd.DataFrame(demand_results)], ignore_index=True)
all_results.to_csv(ROOT / 'artifacts' / 'ml_results.csv', index=False)
all_results

## Selection rule

Select the model on validation metrics, then fit it on train plus validation and report the untouched test result. Keep the full preprocessing pipeline with the model so the deployment layer receives the same feature transformations used during evaluation.